# Trabajo Final ICD 2026 - IBM HR Analytics

Objetivo: construir y comparar dos modelos supervisados para predecir si un empleado abandona la empresa (`Attrition`).

Modelos elegidos:

- Regresion Logistica
- Random Forest

## 1. Carga de librerias y dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")

In [ ]:
DATA_PATH = "../IBM HR Analytics Employee_TF.csv"

df = pd.read_csv(DATA_PATH, sep=";")
df.head()

## 2. Exploracion inicial

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df["Attrition"].value_counts()

## 3. Graficos exploratorios obligatorios

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Attrition", y="MonthlyIncome")
plt.title("Boxplot de salario mensual segun rotacion")
plt.xlabel("Rotacion laboral")
plt.ylabel("Ingreso mensual")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x="Attrition", y="Age")
plt.title("Violin plot de edad segun rotacion")
plt.xlabel("Rotacion laboral")
plt.ylabel("Edad")
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=["int64", "float64"])

plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Mapa de calor de correlaciones numericas")
plt.show()

## 4. Limpieza y preparacion de datos

In [ ]:
target = "Attrition"

columns_to_drop = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]

X = df.drop(columns=[target] + columns_to_drop)
y = df[target].map({"No": 0, "Yes": 1})

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features, categorical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 5. Entrenamiento de modelos

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
    ]
)

models = {
    "Regresion Logistica": logistic_model,
    "Random Forest": random_forest_model,
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Modelo entrenado: {name}")

## 6. Evaluacion y matrices de confusion

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({"Modelo": name, "Accuracy": acc})

    print("=" * 60)
    print(name)
    print("Accuracy:", round(acc, 4))
    print(classification_report(y_test, y_pred, target_names=["No", "Yes"]))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(f"Matriz de confusion - {name}")
    plt.xlabel("Prediccion")
    plt.ylabel("Valor real")
    plt.show()

pd.DataFrame(results)

## 7. Comparacion final

En esta seccion se escribira la comparacion entre modelos una vez ejecutado el notebook.